# 反向传播算法实现

反向传播是训练神经网络的核心算法，通过链式法则高效计算梯度。本notebook演示从零开始实现反向传播。

## 1. 导入必要的库

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

## 2. 实现计算图节点基类

In [ ]:
class Node:
    """计算图中的节点"""
    
    def __init__(self, inputs=[]):
        self.inputs = inputs
        self.outputs = []
        self.value = None
        self.gradients = {}
        
        # 将当前节点添加到输入节点的输出列表
        for node in inputs:
            node.outputs.append(self)
    
    def forward(self):
        """前向传播"""
        raise NotImplementedError
    
    def backward(self):
        """反向传播"""
        raise NotImplementedError

## 3. 实现输入节点

In [ ]:
class Input(Node):
    """输入节点"""
    
    def __init__(self):
        Node.__init__(self)
    
    def forward(self, value=None):
        """设置输入值"""
        if value is not None:
            self.value = value
    
    def backward(self):
        """输入节点收集梯度"""
        self.gradients = {self: 0}
        for n in self.outputs:
            self.gradients[self] += n.gradients[self]

## 4. 实现线性层节点

线性变换：$y = Wx + b$

In [ ]:
class Linear(Node):
    """线性变换节点"""
    
    def __init__(self, X, W, b):
        Node.__init__(self, [X, W, b])
    
    def forward(self):
        """前向传播：y = Wx + b"""
        X = self.inputs[0].value
        W = self.inputs[1].value
        b = self.inputs[2].value
        self.value = np.dot(X, W) + b
    
    def backward(self):
        """反向传播"""
        self.gradients = {n: np.zeros_like(n.value) for n in self.inputs}
        
        for n in self.outputs:
            grad_cost = n.gradients[self]
            # dL/dX = dL/dy * W^T
            self.gradients[self.inputs[0]] += np.dot(grad_cost, self.inputs[1].value.T)
            # dL/dW = X^T * dL/dy
            self.gradients[self.inputs[1]] += np.dot(self.inputs[0].value.T, grad_cost)
            # dL/db = sum(dL/dy)
            self.gradients[self.inputs[2]] += np.sum(grad_cost, axis=0, keepdims=False)

## 5. 实现Sigmoid激活函数节点

Sigmoid: $\sigma(x) = \frac{1}{1 + e^{-x}}$

导数: $\sigma'(x) = \sigma(x)(1 - \sigma(x))$

In [ ]:
class Sigmoid(Node):
    """Sigmoid激活函数"""
    
    def __init__(self, node):
        Node.__init__(self, [node])
    
    def _sigmoid(self, x):
        """Sigmoid函数"""
        return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))
    
    def forward(self):
        """前向传播"""
        self.value = self._sigmoid(self.inputs[0].value)
    
    def backward(self):
        """反向传播"""
        self.gradients = {n: np.zeros_like(n.value) for n in self.inputs}
        
        for n in self.outputs:
            grad_cost = n.gradients[self]
            sigmoid = self.value
            # sigmoid'(x) = sigmoid(x) * (1 - sigmoid(x))
            self.gradients[self.inputs[0]] += sigmoid * (1 - sigmoid) * grad_cost

## 6. 实现MSE损失节点

均方误差：$L = \frac{1}{n}\sum(y_{pred} - y_{true})^2$

In [ ]:
class MSE(Node):
    """均方误差损失"""
    
    def __init__(self, y_pred, y_true):
        Node.__init__(self, [y_pred, y_true])
    
    def forward(self):
        """计算MSE"""
        y_pred = self.inputs[0].value
        y_true = self.inputs[1].value
        self.diff = y_pred - y_true
        self.value = np.mean(self.diff ** 2)
    
    def backward(self):
        """反向传播"""
        n = self.diff.shape[0]
        self.gradients[self.inputs[0]] = 2 * self.diff / n
        self.gradients[self.inputs[1]] = -2 * self.diff / n

## 7. 实现拓扑排序

对计算图进行拓扑排序，确保前向和反向传播的正确顺序。

In [ ]:
def topological_sort(feed_dict):
    """拓扑排序计算图"""
    input_nodes = [n for n in feed_dict.keys()]
    
    G = {}
    nodes = [n for n in input_nodes]
    
    while len(nodes) > 0:
        n = nodes.pop(0)
        if n not in G:
            G[n] = {'in': set(), 'out': set()}
        for m in n.outputs:
            if m not in G:
                G[m] = {'in': set(), 'out': set()}
            G[n]['out'].add(m)
            G[m]['in'].add(n)
            nodes.append(m)
    
    L = []
    S = set(input_nodes)
    
    while len(S) > 0:
        n = S.pop()
        if isinstance(n, Input):
            n.value = feed_dict[n]
        L.append(n)
        
        for m in n.outputs:
            G[n]['out'].remove(m)
            G[m]['in'].remove(n)
            if len(G[m]['in']) == 0:
                S.add(m)
    
    return L

## 8. 测试计算图

构建简单的计算图：$y = \sigma(Wx + b)$

In [ ]:
# 创建计算图节点
X = Input()
W = Input()
b = Input()

linear = Linear(X, W, b)
output = Sigmoid(linear)

# 设置输入值
feed_dict = {
    X: np.array([[1.0, 2.0]]),
    W: np.array([[0.5], [0.3]]),
    b: np.array([[0.1]])
}

# 拓扑排序
graph = topological_sort(feed_dict)

# 前向传播
for node in graph:
    node.forward()

print("计算图前向传播结果：")
print(f"输入 X: {feed_dict[X]}")
print(f"权重 W: {feed_dict[W].flatten()}")
print(f"偏置 b: {feed_dict[b].flatten()}")
print(f"线性输出: {linear.value}")
print(f"Sigmoid输出: {output.value}")

## 9. 实现完整的神经网络类

In [ ]:
class NeuralNetwork:
    """简单的全连接神经网络"""
    
    def __init__(self, layer_sizes, learning_rate=0.1):
        self.layer_sizes = layer_sizes
        self.learning_rate = learning_rate
        
        # 初始化权重和偏置
        self.weights = []
        self.biases = []
        
        for i in range(len(layer_sizes) - 1):
            # He初始化
            W = np.random.randn(layer_sizes[i], layer_sizes[i+1]) * np.sqrt(2.0/layer_sizes[i])
            b = np.zeros((1, layer_sizes[i+1]))
            self.weights.append(W)
            self.biases.append(b)
    
    def sigmoid(self, x):
        """Sigmoid激活函数"""
        return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))
    
    def forward(self, X):
        """前向传播"""
        self.activations = [X]
        
        for i in range(len(self.weights)):
            z = np.dot(self.activations[-1], self.weights[i]) + self.biases[i]
            if i < len(self.weights) - 1:
                a = self.sigmoid(z)
            else:
                a = z  # 输出层线性输出
            self.activations.append(a)
        
        return self.activations[-1]
    
    def backward(self, X, y):
        """反向传播"""
        m = X.shape[0]
        
        # 输出层梯度
        delta = self.activations[-1] - y
        
        weight_grads = []
        bias_grads = []
        
        # 从后向前计算梯度
        for i in range(len(self.weights) - 1, -1, -1):
            dW = np.dot(self.activations[i].T, delta) / m
            db = np.sum(delta, axis=0, keepdims=True) / m
            
            weight_grads.insert(0, dW)
            bias_grads.insert(0, db)
            
            if i > 0:
                # 计算前一层的delta
                delta = np.dot(delta, self.weights[i].T) * \
                        self.activations[i] * (1 - self.activations[i])
        
        return weight_grads, bias_grads
    
    def train_step(self, X, y):
        """训练一步"""
        # 前向传播
        y_pred = self.forward(X)
        
        # 计算损失
        loss = np.mean((y_pred - y) ** 2)
        
        # 反向传播
        weight_grads, bias_grads = self.backward(X, y)
        
        # 更新参数
        for i in range(len(self.weights)):
            self.weights[i] -= self.learning_rate * weight_grads[i]
            self.biases[i] -= self.learning_rate * bias_grads[i]
        
        return loss
    
    def train(self, X, y, epochs=1000, verbose=True):
        """训练网络"""
        losses = []
        
        for epoch in range(epochs):
            loss = self.train_step(X, y)
            losses.append(loss)
            
            if verbose and (epoch + 1) % 200 == 0:
                print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.6f}")
        
        return losses
    
    def predict(self, X):
        """预测"""
        return self.forward(X)

## 10. 测试：XOR问题

XOR是一个经典的非线性问题，需要隐藏层才能解决。

In [ ]:
# 创建XOR数据
X_train = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_train = np.array([[0], [1], [1], [0]])

print("XOR问题数据：")
print("输入：")
print(X_train)
print("\n输出：")
print(y_train.flatten())

In [ ]:
# 创建神经网络
nn = NeuralNetwork([2, 4, 1], learning_rate=0.5)

print("\n网络结构: 输入层(2) -> 隐藏层(4) -> 输出层(1)")
print("开始训练...\n")

# 训练
losses = nn.train(X_train, y_train, epochs=2000)

In [ ]:
# 测试结果
print("\n训练后的预测结果：")
predictions = nn.predict(X_train)

for i in range(len(X_train)):
    pred_value = predictions[i][0]
    pred_class = 1 if pred_value > 0.5 else 0
    print(f"输入: {X_train[i]}, 真实值: {y_train[i][0]}, "
          f"预测值: {pred_value:.4f}, 预测类别: {pred_class}")

## 11. 可视化训练过程

In [ ]:
# 绘制损失曲线和决策边界
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 损失曲线
ax1.plot(losses, linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('训练损失曲线')
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# 决策边界
h = 0.01
x_min, x_max = -0.5, 1.5
y_min, y_max = -0.5, 1.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))
Z = nn.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

ax2.contourf(xx, yy, Z, alpha=0.8, cmap=plt.cm.RdYlBu)
ax2.scatter(X_train[:, 0], X_train[:, 1], c=y_train.ravel(),
           s=200, edgecolors='k', cmap=plt.cm.RdYlBu, linewidth=2)
ax2.set_xlabel('x1')
ax2.set_ylabel('x2')
ax2.set_title('XOR决策边界')
ax2.set_xlim(x_min, x_max)
ax2.set_ylim(y_min, y_max)

plt.tight_layout()
plt.show()

## 12. 测试：函数拟合

使用神经网络拟合sin函数。

In [ ]:
# 生成sin函数数据
X_reg = np.linspace(0, 2*np.pi, 100).reshape(-1, 1)
y_reg = np.sin(X_reg)

# 创建深层网络
nn_reg = NeuralNetwork([1, 10, 10, 1], learning_rate=0.1)

print("网络结构: 输入层(1) -> 隐藏层(10) -> 隐藏层(10) -> 输出层(1)")
print("训练中...\n")

# 训练
losses_reg = nn_reg.train(X_reg, y_reg, epochs=1000, verbose=True)

In [ ]:
# 预测
y_pred = nn_reg.predict(X_reg)

# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 损失曲线
ax1.plot(losses_reg, linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('训练损失')
ax1.grid(True, alpha=0.3)

# 拟合结果
ax2.plot(X_reg, y_reg, 'b-', label='真实值', linewidth=2)
ax2.plot(X_reg, y_pred, 'r--', label='预测值', linewidth=2)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_title('sin函数拟合')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n最终损失: {losses_reg[-1]:.6f}")

## 13. 可视化激活函数及其导数

In [ ]:
# 绘制激活函数
x = np.linspace(-5, 5, 100)

# Sigmoid
sigmoid = 1 / (1 + np.exp(-x))
sigmoid_grad = sigmoid * (1 - sigmoid)

# Tanh
tanh = np.tanh(x)
tanh_grad = 1 - tanh**2

# ReLU
relu = np.maximum(0, x)
relu_grad = (x > 0).astype(float)

# 绘图
fig, axes = plt.subplots(3, 2, figsize=(12, 10))

# Sigmoid
axes[0, 0].plot(x, sigmoid, 'b-', linewidth=2)
axes[0, 0].set_title('Sigmoid函数')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(x, sigmoid_grad, 'r-', linewidth=2)
axes[0, 1].set_title('Sigmoid导数')
axes[0, 1].grid(True, alpha=0.3)

# Tanh
axes[1, 0].plot(x, tanh, 'b-', linewidth=2)
axes[1, 0].set_title('Tanh函数')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(x, tanh_grad, 'r-', linewidth=2)
axes[1, 1].set_title('Tanh导数')
axes[1, 1].grid(True, alpha=0.3)

# ReLU
axes[2, 0].plot(x, relu, 'b-', linewidth=2)
axes[2, 0].set_title('ReLU函数')
axes[2, 0].grid(True, alpha=0.3)

axes[2, 1].plot(x, relu_grad, 'r-', linewidth=2)
axes[2, 1].set_title('ReLU导数')
axes[2, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 14. 梯度检查

使用数值梯度验证反向传播实现的正确性。

In [ ]:
def numerical_gradient(f, x, h=1e-5):
    """数值方法计算梯度"""
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    
    while not it.finished:
        idx = it.multi_index
        old_value = x[idx]
        
        x[idx] = old_value + h
        fxh1 = f(x)
        
        x[idx] = old_value - h
        fxh2 = f(x)
        
        grad[idx] = (fxh1 - fxh2) / (2 * h)
        x[idx] = old_value
        it.iternext()
    
    return grad

# 简单测试
def test_loss(W):
    X = np.array([[1.0, 2.0]])
    y = np.array([[1.0]])
    y_pred = np.dot(X, W.reshape(-1, 1))
    return np.mean((y_pred - y)**2)

W_test = np.array([0.5, 0.3])
num_grad = numerical_gradient(test_loss, W_test)

# 解析梯度
X = np.array([[1.0, 2.0]])
y = np.array([[1.0]])
y_pred = np.dot(X, W_test.reshape(-1, 1))
anal_grad = 2 * X.T.dot(y_pred - y) / X.shape[0]

print("梯度检查：")
print(f"数值梯度: {num_grad}")
print(f"解析梯度: {anal_grad.flatten()}")
print(f"差异: {np.abs(num_grad - anal_grad.flatten())}")

## 15. 总结

通过本notebook，我们从零开始实现了反向传播算法：

1. **计算图**：实现了支持前向和反向传播的计算图节点
2. **神经网络**：构建了完整的全连接神经网络
3. **XOR问题**：成功解决了非线性分类问题
4. **函数拟合**：展示了神经网络的函数逼近能力
5. **梯度检查**：验证了反向传播实现的正确性

**关键要点**：
- 反向传播的核心是链式法则
- 计算图使得梯度计算更加清晰
- 激活函数的选择对训练效果影响很大
- 梯度检查是验证实现正确性的重要工具